# Sandbox — Silver Geolocation

Exploração por trás de `silver_rules.tratar_geolocation`. Esta é a maior tabela do dataset (~1 milhão de linhas) e tem uma decisão específica: **deduplicação**. O arquivo original repete a mesma combinação (CEP, lat, lng, cidade, UF) muitas vezes, então aplicamos `dropDuplicates` para reduzir o volume sem perder informação.

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import col, trim, lower, upper, current_timestamp

spark = create_spark_session('Sandbox-Silver-Geolocation')

df_bronze = spark.read.parquet('s3a://bronze/olist/geolocation')
df_bronze.show(5, truncate=False)
df_bronze.printSchema()

Medindo o impacto do `dropDuplicates`: contamos antes e depois para quantificar quanta redundância existe no arquivo bruto.

In [ ]:
total_antes = df_bronze.count()
total_depois = df_bronze.dropDuplicates().count()

print('Linhas antes do dedup:', total_antes)
print('Linhas depois do dedup:', total_depois)
print('Reducao:', round(100 * (1 - total_depois / total_antes), 1), '%')

In [ ]:
df_silver = (df_bronze
    .withColumn('geolocation_zip_code_prefix', trim(col('geolocation_zip_code_prefix')))
    .withColumn('geolocation_lat', col('geolocation_lat').cast('double'))
    .withColumn('geolocation_lng', col('geolocation_lng').cast('double'))
    .withColumn('geolocation_city', lower(trim(col('geolocation_city'))))
    .withColumn('geolocation_state', upper(trim(col('geolocation_state'))))
    .dropDuplicates()
    .withColumn('dt_criacao_silver', current_timestamp()))

df_silver.printSchema()
df_silver.show(5, truncate=False)

**Lógica promovida para `silver_rules.tratar_geolocation`.**

In [ ]:
spark.stop()